# 🛡️ AI SHIELD — Complete ML Pipeline
### Smart Handling of Integrity in Ethical Live Dialogue
**Program:** PIJAK in collaboration with IBM SkillsBuild — Dicoding  
**Model:** IndoBERT (`indobenchmark/indobert-base-p1`) — Binary Text Classification  
**Dataset:** [id-multi-label-hate-speech-and-abusive-language-detection](https://github.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection)

---

## 📋 Alur Pipeline
| Tahap | Deskripsi |
|-------|----------|
| **TAHAP 1** | Install Library & Setup Lingkungan |
| **TAHAP 2** | Download Dataset dari GitHub |
| **TAHAP 3** | Exploratory Data Analysis (EDA) |
| **TAHAP 4** | Preprocessing & Relabeling ke 2 Kelas Biner |
| **TAHAP 5** | Split Dataset: Train 70% · Val 15% · Test 15% |
| **TAHAP 6** | Fine-tuning IndoBERT |
| **TAHAP 7** | Evaluasi Model (Accuracy, F1, Confusion Matrix, ROC) |
| **TAHAP 8** | Kalibrasi Confidence Threshold |
| **TAHAP 9** | Fungsi Inferensi `predict()` + Export untuk Backend |
| **TAHAP 10** | Simpan Model ke Google Drive |

> ⚠️ **PENTING:** Aktifkan GPU sebelum menjalankan notebook ini!  
> Caranya: **Runtime → Change runtime type → T4 GPU → Save**  
> Estimasi waktu training: **~30–60 menit** dengan GPU T4

---
## ⚙️ TAHAP 1 — Install Library & Setup Lingkungan

In [ ]:
# Install semua library yang dibutuhkan
print('⏳ Menginstall library... (1-3 menit pertama kali)')
!pip install transformers==4.40.0 datasets==2.19.0 accelerate==0.29.3 -q
!pip install scikit-learn pandas numpy matplotlib seaborn requests -q
print('✅ Instalasi selesai!')

In [ ]:
# Import semua library
import os, re, json, time, warnings, requests, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Deteksi device (GPU/CPU)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('=' * 55)
print('🔧 KONFIGURASI LINGKUNGAN')
print('=' * 55)
print(f'  PyTorch versi  : {torch.__version__}')
print(f'  Device         : {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU            : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM           : {vram:.1f} GB')
else:
    print('  ⚠️  GPU tidak terdeteksi!')
    print('     → Aktifkan GPU: Runtime → Change runtime type → T4 GPU')
print('=' * 55)

---
## 📥 TAHAP 2 — Download Dataset dari GitHub

Dataset: **id-multi-label-hate-speech-and-abusive-language-detection** oleh okkyibrohim  
Repo: https://github.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection

File yang diunduh:
- `Re_Dataset.csv` → Dataset utama (~13.000 tweet)
- `new_kamusalay.csv` → Kamus normalisasi kata alay/slang
- `stopwordbahasa.csv` → Stopwords bahasa Indonesia

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/master/'
FILES = ['Re_Dataset.csv', 'new_kamusalay.csv', 'stopwordbahasa.csv']

os.makedirs('data', exist_ok=True)

for fname in FILES:
    fpath = f'data/{fname}'
    if not os.path.exists(fpath):
        print(f'📥 Mengunduh {fname}...')
        r = requests.get(BASE_URL + fname, timeout=60)
        if r.status_code == 200:
            with open(fpath, 'wb') as f:
                f.write(r.content)
            print(f'   ✅ Berhasil — {os.path.getsize(fpath)/1024:.1f} KB')
        else:
            print(f'   ❌ Gagal (status: {r.status_code})')
    else:
        print(f'   ⏭️  {fname} sudah ada.')

# Load dataset
df = pd.read_csv('data/Re_Dataset.csv', encoding='utf-8')

# Load kamus alay
alay_df = pd.read_csv('data/new_kamusalay.csv', encoding='latin-1', header=None)
alay_df.columns = ['original', 'replacement']
ALAY_MAP = dict(zip(alay_df['original'].str.lower(), alay_df['replacement']))

print(f'\n📊 Dataset dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom')
print(f'📖 Kamus alay: {len(ALAY_MAP):,} kata')
print(f'\n📋 Kolom: {list(df.columns)}')
print()
print('5 Baris Pertama:')
df.head()

---
## 📊 TAHAP 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ─── Informasi dasar dataset ───
print('📋 INFO DATASET:')
df.info()
print()

# Missing values
missing = df.isnull().sum()
if missing.sum() == 0:
    print('✅ Tidak ada missing value!')
else:
    print('⚠️  Missing values:')
    print(missing[missing > 0])

# Duplikasi
n_dup = df.duplicated(subset='Tweet').sum()
print(f'\n🔁 Duplikat: {n_dup:,}')

# Distribusi label
print(f'\n📊 Distribusi Label:')
label_cols = [c for c in df.columns if c != 'Tweet']
print(df[label_cols].sum().sort_values(ascending=False).to_string())

In [ ]:
# ─── Buat label biner untuk preview ───
df['binary_label'] = ((df['HS'] == 1) | (df['Abusive'] == 1)).astype(int)
df['text_length']  = df['Tweet'].astype(str).apply(len)
df['word_count']   = df['Tweet'].astype(str).apply(lambda x: len(x.split()))

LABEL_NAMES = {0: 'PANTAS', 1: 'TIDAK PANTAS'}
label_counts = df['binary_label'].value_counts().sort_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis — AI SHIELD Dataset', fontsize=15, fontweight='bold')
colors = ['#27AE60', '#E74C3C']

# 1. Distribusi label biner (bar)
bars = axes[0,0].bar(
    [LABEL_NAMES[i] for i in label_counts.index],
    label_counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5
)
for bar in bars:
    h = bar.get_height()
    axes[0,0].text(bar.get_x() + bar.get_width()/2, h + 50,
                   f'{h:,}\n({h/len(df)*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[0,0].set_title('Distribusi Label Biner', fontweight='bold')
axes[0,0].set_ylabel('Jumlah Tweet')
axes[0,0].set_ylim(0, label_counts.max() * 1.2)
axes[0,0].spines['top'].set_visible(False)
axes[0,0].spines['right'].set_visible(False)

# 2. Pie chart
axes[0,1].pie(
    label_counts.values,
    labels=[LABEL_NAMES[i] for i in label_counts.index],
    colors=colors, autopct='%1.1f%%', startangle=90,
    wedgeprops={'linewidth': 2, 'edgecolor': 'white'},
    textprops={'fontsize': 12}
)
axes[0,1].set_title('Proporsi Label Biner', fontweight='bold')

# 3. Distribusi panjang teks
axes[1,0].hist(df['text_length'], bins=40, color='#9B59B6', alpha=0.8, edgecolor='white')
axes[1,0].axvline(df['text_length'].mean(), color='red', linestyle='--', linewidth=2,
                  label=f'Rata-rata: {df["text_length"].mean():.0f} karakter')
axes[1,0].set_title('Distribusi Panjang Karakter', fontweight='bold')
axes[1,0].set_xlabel('Jumlah Karakter')
axes[1,0].set_ylabel('Frekuensi')
axes[1,0].legend()
axes[1,0].spines['top'].set_visible(False)
axes[1,0].spines['right'].set_visible(False)

# 4. Distribusi jumlah kata
axes[1,1].hist(df['word_count'], bins=30, color='#1ABC9C', alpha=0.8, edgecolor='white')
axes[1,1].axvline(df['word_count'].mean(), color='red', linestyle='--', linewidth=2,
                  label=f'Rata-rata: {df["word_count"].mean():.0f} kata')
axes[1,1].set_title('Distribusi Jumlah Kata', fontweight='bold')
axes[1,1].set_xlabel('Jumlah Kata')
axes[1,1].set_ylabel('Frekuensi')
axes[1,1].legend()
axes[1,1].spines['top'].set_visible(False)
axes[1,1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('data/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Statistik Dataset:')
print(f'  Total tweet          : {len(df):,}')
print(f'  PANTAS               : {(df["binary_label"]==0).sum():,} ({(df["binary_label"]==0).mean()*100:.1f}%)')
print(f'  TIDAK PANTAS         : {(df["binary_label"]==1).sum():,} ({(df["binary_label"]==1).mean()*100:.1f}%)')
print(f'  Rata-rata karakter   : {df["text_length"].mean():.0f}')
print(f'  Rata-rata kata       : {df["word_count"].mean():.0f}')

In [ ]:
# ─── Contoh sampel tweet ───
print('📝 CONTOH TWEET — Label PANTAS:')
print('─' * 65)
for _, row in df[df['binary_label'] == 0].sample(4, random_state=42).iterrows():
    print(f'  ✅ {row["Tweet"][:90]}')

print()
print('🚫 CONTOH TWEET — Label TIDAK PANTAS:')
print('─' * 65)
for _, row in df[df['binary_label'] == 1].sample(4, random_state=42).iterrows():
    print(f'  🚫 {row["Tweet"][:90]}')

---
## 🧹 TAHAP 4 — Preprocessing & Relabeling ke 2 Kelas Biner

### Strategi Preprocessing:
1. **Lowercase** — ubah semua ke huruf kecil
2. **Hapus noise** — RT, @mention, URL, emoji, angka, tanda baca
3. **Normalisasi alay** — konversi slang ke kata baku via kamus
4. **Hapus karakter berulang** dan spasi berlebih

### Strategi Relabeling:
| Kondisi Label Asli | Label Baru |
|--------------------|------------|
| `HS == 0` **DAN** `Abusive == 0` | `PANTAS` (0) |
| `HS == 1` **ATAU** `Abusive == 1` | `TIDAK PANTAS` (1) |

In [ ]:
def preprocess_text(text: str, alay_mapping: dict = None) -> str:
    """
    Membersihkan teks tweet untuk keperluan klasifikasi.

    Args:
        text          : Teks mentah
        alay_mapping  : Kamus normalisasi kata alay (dict)

    Returns:
        str: Teks yang sudah dibersihkan
    """
    if not isinstance(text, str):
        return ''

    # 1. Lowercase
    text = text.lower()

    # 2. Hapus RT
    text = re.sub(r'\brt\b', '', text)

    # 3. Hapus mention (@username)
    text = re.sub(r'@[\w]+', '', text)

    # 4. Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 5. Hapus simbol hashtag
    text = re.sub(r'#', '', text)

    # 6. Hapus emoji & karakter non-ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 7. Hapus angka
    text = re.sub(r'\d+', '', text)

    # 8. Hapus tanda baca (pertahankan spasi)
    text = re.sub(r'[^\w\s]', ' ', text)

    # 9. Normalisasi kata alay
    if alay_mapping:
        words = text.split()
        words = [alay_mapping.get(w, w) for w in words]
        text  = ' '.join(words)

    # 10. Hapus karakter berulang (hahahaha → haha)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 11. Rapikan spasi
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Demo preprocessing
demos = [
    'RT @user123: Hei kamu bego bgt sih!! Goblok https://t.co/abc123',
    'wkwkwk mantul bro 😂😂 #BanggaIndonesia',
    'gue benci banget sama lo anjg*k kampret!!'
]

print('📋 Demo Preprocessing:')
print('─' * 70)
for t in demos:
    cleaned = preprocess_text(t, ALAY_MAP)
    print(f'  SEBELUM : {t}')
    print(f'  SESUDAH : {cleaned}')
    print('─' * 70)

In [ ]:
# ─── Terapkan preprocessing ke seluruh dataset ───
print('⏳ Menerapkan preprocessing ke seluruh dataset...')

df['clean_text'] = df['Tweet'].apply(lambda x: preprocess_text(x, ALAY_MAP))

# Relabeling ke biner
df['label'] = ((df['HS'] == 1) | (df['Abusive'] == 1)).astype(int)

# Hapus baris kosong
n_before = len(df)
df = df[df['clean_text'].str.len() > 2].reset_index(drop=True)
n_after  = len(df)

print(f'✅ Preprocessing selesai!')
print(f'   Sebelum  : {n_before:,} baris')
print(f'   Sesudah  : {n_after:,} baris')
print(f'   Dihapus  : {n_before - n_after} baris (teks kosong)')

# Distribusi akhir
print(f'\n📊 Distribusi Label Biner:')
for k, v in df['label'].value_counts().sort_index().items():
    bar = '█' * int(v / len(df) * 30)
    print(f'  {LABEL_NAMES[k]:15s} | {bar} {v:,} ({v/len(df)*100:.1f}%)')

# Tampilkan perbandingan sebelum-sesudah
print('\n📋 Contoh Perbandingan:')
for _, row in df[['Tweet', 'clean_text', 'label']].sample(3, random_state=1).iterrows():
    print(f'  ASLI   : {str(row["Tweet"])[:80]}')
    print(f'  BERSIH : {row["clean_text"]}')
    print(f'  LABEL  : {LABEL_NAMES[row["label"]]}')
    print()

---
## ✂️ TAHAP 5 — Split Dataset (Train / Validation / Test)

**Rasio:** Train 70% · Validation 15% · Test 15%  
**Metode:** Stratified split (proporsi kelas seimbang di setiap subset)

In [ ]:
X = df['clean_text'].values
y = df['label'].values

# Split 1: train (70%) vs temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Split 2: val (15%) vs test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Buat DataFrame
df_train = pd.DataFrame({'text': X_train, 'label': y_train})
df_val   = pd.DataFrame({'text': X_val,   'label': y_val})
df_test  = pd.DataFrame({'text': X_test,  'label': y_test})

# Simpan CSV
df_train.to_csv('data/train.csv', index=False, encoding='utf-8')
df_val.to_csv('data/val.csv',     index=False, encoding='utf-8')
df_test.to_csv('data/test.csv',   index=False, encoding='utf-8')

# Ringkasan
print('✅ Dataset berhasil dibagi!')
print()
print(f'  {"Split":<12} {"Total":>8} {"PANTAS":>12} {"TIDAK PANTAS":>14}')
print('  ' + '─' * 50)
for name, data_y in [('Train (70%)', y_train), ('Val (15%)', y_val), ('Test (15%)', y_test)]:
    n  = len(data_y)
    n0 = (data_y == 0).sum()
    n1 = (data_y == 1).sum()
    print(f'  {name:<12} {n:>8,} {n0:>6,} ({n0/n*100:.0f}%) {n1:>6,} ({n1/n*100:.0f}%)')
print('  ' + '─' * 50)
print(f'  {"TOTAL":<12} {len(y):>8,} {(y==0).sum():>6,} ({(y==0).mean()*100:.0f}%) {(y==1).sum():>6,} ({(y==1).mean()*100:.0f}%)')

---
## 🧠 TAHAP 6 — Fine-tuning IndoBERT

**Model Base:** `indobenchmark/indobert-base-p1`  
**Task:** Binary Sequence Classification (PANTAS / TIDAK PANTAS)  
**Framework:** PyTorch + HuggingFace Trainer API

> ⏳ **Estimasi waktu:** ~30–60 menit dengan GPU T4 Colab

In [ ]:
# ─── Konfigurasi model ───
MODEL_NAME    = 'indobenchmark/indobert-base-p1'
MAX_LENGTH    = 128
NUM_LABELS    = 2
MODEL_LABELS  = ['PANTAS', 'TIDAK PANTAS']

# Load tokenizer
print(f'📥 Memuat tokenizer: {MODEL_NAME}')
print('   (Proses pertama kali ~1-2 menit...)')
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

print(f'✅ Tokenizer siap!')
print(f'   Vocab size  : {tokenizer.vocab_size:,}')
print(f'   Max length  : {MAX_LENGTH}')

# Demo tokenisasi
sample = 'hei kamu jangan berlaku kasar sama orang lain ya'
toks = tokenizer(sample, max_length=MAX_LENGTH, truncation=True)
print(f'\n🔤 Demo Tokenisasi:')
print(f'   Teks   : {sample}')
print(f'   Tokens : {tokenizer.convert_ids_to_tokens(toks["input_ids"])}')

In [ ]:
class HateSpeechDataset(Dataset):
    """Custom PyTorch Dataset untuk fine-tuning IndoBERT."""

    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = list(texts)
        self.labels     = list(labels)
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'token_type_ids' : encoding.get(
                'token_type_ids',
                torch.zeros(self.max_length, dtype=torch.long)
            ).squeeze(),
            'labels'         : torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }


train_dataset = HateSpeechDataset(df_train['text'], df_train['label'], tokenizer, MAX_LENGTH)
val_dataset   = HateSpeechDataset(df_val['text'],   df_val['label'],   tokenizer, MAX_LENGTH)
test_dataset  = HateSpeechDataset(df_test['text'],  df_test['label'],  tokenizer, MAX_LENGTH)

print('✅ Dataset objects siap!')
print(f'   Train   : {len(train_dataset):,}')
print(f'   Val     : {len(val_dataset):,}')
print(f'   Test    : {len(test_dataset):,}')

In [ ]:
# ─── Load model ───
print(f'📥 Memuat model: {MODEL_NAME}')
print('   (Proses pertama kali ~2-4 menit...)')

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={i: l for i, l in enumerate(MODEL_LABELS)},
    label2id={l: i for i, l in enumerate(MODEL_LABELS)}
)
model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ Model dimuat ke {DEVICE}!')
print(f'   Total parameter     : {total_params:,}')
print(f'   Parameter trainable : {trainable_params:,}')

In [ ]:
# ─── Fungsi metrik ───
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    return {
        'accuracy'        : accuracy_score(labels, predictions),
        'f1_macro'        : f1_score(labels, predictions, average='macro'),
        'f1_tidak_pantas' : f1_score(labels, predictions, pos_label=1, average='binary'),
        'precision_macro' : precision_score(labels, predictions, average='macro'),
        'recall_macro'    : recall_score(labels, predictions, average='macro'),
    }


# ─── Konfigurasi Training ───
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 5
OUTPUT_DIR    = './model_output'

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LEARNING_RATE,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    logging_strategy            = 'epoch',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_tidak_pantas',
    greater_is_better           = True,
    report_to                   = 'none',
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 0,
    seed                        = 42,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print('✅ Trainer siap!')
print(f'   Epochs         : {NUM_EPOCHS}')
print(f'   Batch size     : {BATCH_SIZE}')
print(f'   Learning rate  : {LEARNING_RATE}')
print(f'   Mixed precision: {torch.cuda.is_available()} (FP16)')
print(f'   Best metric    : f1_tidak_pantas (target ≥ 0.82)')
print()
print('🚀 Siap training! Jalankan cell berikutnya...')

In [ ]:
# ─── MULAI TRAINING ─────────────────────────────────────────────────
print('🚀 Memulai Fine-tuning IndoBERT...')
print(f'   Model  : {MODEL_NAME}')
print(f'   Device : {DEVICE}')
print(f'   Epochs : {NUM_EPOCHS}')
if DEVICE.type == 'cpu':
    print('   ⚠️  Training di CPU akan SANGAT LAMBAT!')
    print('      Aktifkan GPU: Runtime → Change runtime type → T4 GPU')
print()

train_result = trainer.train()

print()
print('🎉 Training SELESAI!')
print(f'   Waktu training : {train_result.metrics["train_runtime"]:.0f} detik '
      f'({train_result.metrics["train_runtime"]/60:.1f} menit)')
print(f'   Train loss     : {train_result.metrics["train_loss"]:.4f}')

In [ ]:
# ─── Evaluasi pada Validation Set ───
print('📊 Evaluasi pada Validation Set:')
val_results = trainer.evaluate()
print()
for k, v in val_results.items():
    if not k.startswith('epoch'):
        print(f'   {k:<35}: {v:.4f}')

In [ ]:
# ─── Simpan Model ───
SAVE_DIR = './indobert_aishield'

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Simpan konfigurasi
config_info = {
    'model_name'           : MODEL_NAME,
    'num_labels'           : NUM_LABELS,
    'label_names'          : MODEL_LABELS,
    'max_length'           : MAX_LENGTH,
    'num_epochs'           : NUM_EPOCHS,
    'learning_rate'        : LEARNING_RATE,
    'batch_size'           : BATCH_SIZE,
    'confidence_threshold' : 0.75,
    'val_metrics'          : {k: round(v, 4) for k, v in val_results.items()}
}
with open(f'{SAVE_DIR}/model_config.json', 'w') as f:
    json.dump(config_info, f, indent=2)

print('✅ Model berhasil disimpan!')
print(f'   Lokasi : {SAVE_DIR}/')
print(f'   File   :')
for fname in sorted(os.listdir(SAVE_DIR)):
    size_mb = os.path.getsize(f'{SAVE_DIR}/{fname}') / 1e6
    print(f'            {fname} ({size_mb:.1f} MB)')

---
## 📈 TAHAP 7 — Evaluasi Model pada Test Set

Metrik yang dihitung:
- **Accuracy**, **Precision**, **Recall**, **F1-Score**
- **Confusion Matrix** (absolut & persentase)
- **ROC Curve & AUC**
- **Precision-Recall Curve**

In [ ]:
# ─── Prediksi pada Test Set ───
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

all_labels, all_predictions, all_probs = [], [], []

print('⏳ Menjalankan inferensi pada test set...')
model.eval()
with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        out = model(
            input_ids      = batch['input_ids'].to(DEVICE),
            attention_mask = batch['attention_mask'].to(DEVICE),
            token_type_ids = batch['token_type_ids'].to(DEVICE)
        )
        probs = torch.softmax(out.logits, dim=-1)
        preds = torch.argmax(out.logits, dim=-1)

        all_labels.extend(batch['labels'].numpy())
        all_predictions.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels      = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probs       = np.array(all_probs)
conf_scores     = all_probs[:, 1]  # P(TIDAK PANTAS)

print(f'✅ Inferensi selesai! {len(all_labels):,} sampel diproses.')

In [ ]:
# ─── Metrik Evaluasi ───
acc        = accuracy_score(all_labels, all_predictions)
f1_macro   = f1_score(all_labels, all_predictions, average='macro')
f1_bin     = f1_score(all_labels, all_predictions, pos_label=1, average='binary')
prec_macro = precision_score(all_labels, all_predictions, average='macro')
rec_macro  = recall_score(all_labels, all_predictions, average='macro')

print('=' * 60)
print('📊 HASIL EVALUASI — AI SHIELD IndoBERT (Test Set)')
print('=' * 60)

metrics_table = [
    ('Accuracy',          acc,        0.85, '≥ 85%'),
    ('F1 TIDAK PANTAS',   f1_bin,     0.82, '≥ 82%'),
    ('F1 Macro',          f1_macro,   None, '-'),
    ('Precision Macro',   prec_macro, None, '-'),
    ('Recall Macro',      rec_macro,  None, '-'),
]

print(f'\n  {"Metrik":<22} {"Nilai":>8}   {"Target":>8}   {"Status"}')
print('  ' + '─' * 52)
for name, value, target, target_str in metrics_table:
    status = ''
    if target is not None:
        status = '✅ PASS' if value >= target else '❌ FAIL'
    print(f'  {name:<22} {value:>8.4f}   {target_str:>8}   {status}')

print()
print('  📋 Classification Report:')
print()
report = classification_report(all_labels, all_predictions,
                                target_names=MODEL_LABELS, digits=4)
for line in report.split('\n'):
    print('    ' + line)

In [ ]:
# ─── Confusion Matrix ───
cm      = confusion_matrix(all_labels, all_predictions)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Absolut
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=MODEL_LABELS, yticklabels=MODEL_LABELS,
            ax=axes[0], linewidths=1, linecolor='white',
            annot_kws={'fontsize': 14, 'fontweight': 'bold'})
axes[0].set_title('Confusion Matrix (Absolut)', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Prediksi', fontsize=11)
axes[0].set_ylabel('Aktual', fontsize=11)
axes[0].tick_params(rotation=0)

# Persentase
annot = np.array([[f'{v:.1%}\n({cm[i,j]:,})' for j, v in enumerate(row)]
                  for i, row in enumerate(cm_norm)])
sns.heatmap(cm_norm, annot=annot, fmt='', cmap='RdYlGn',
            xticklabels=MODEL_LABELS, yticklabels=MODEL_LABELS,
            ax=axes[1], linewidths=1, linecolor='white',
            annot_kws={'fontsize': 11}, vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (Persentase)', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Prediksi', fontsize=11)
axes[1].set_ylabel('Aktual', fontsize=11)
axes[1].tick_params(rotation=0)

plt.suptitle('Confusion Matrix — AI SHIELD IndoBERT', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/eval_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print('\n📋 Interpretasi:')
print(f'   TN  : {tn:,} — PANTAS diprediksi PANTAS        ✅')
print(f'   FP  : {fp:,} — PANTAS diprediksi TIDAK PANTAS  ⚠️  (pesan normal tersensor)')
print(f'   FN  : {fn:,} — TIDAK PANTAS diprediksi PANTAS  ⚠️  (pesan kasar lolos)')
print(f'   TP  : {tp:,} — TIDAK PANTAS diprediksi TIDAK PANTAS ✅')

In [ ]:
# ─── ROC Curve & Precision-Recall Curve ───
fpr, tpr, _ = roc_curve(all_labels, conf_scores)
roc_auc     = auc(fpr, tpr)
prec_curve, rec_curve, _ = precision_recall_curve(all_labels, conf_scores)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='blue')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f'ROC Curve (AUC = {roc_auc:.4f})', fontweight='bold')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# PR Curve
axes[1].plot(rec_curve, prec_curve, 'r-', linewidth=2, label='Precision-Recall Curve')
axes[1].fill_between(rec_curve, prec_curve, alpha=0.08, color='red')
axes[1].set_xlabel('Recall (TIDAK PANTAS)')
axes[1].set_ylabel('Precision (TIDAK PANTAS)')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('ROC & Precision-Recall Curve', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/eval_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'📊 ROC AUC: {roc_auc:.4f}')

---
## 🎯 TAHAP 8 — Kalibrasi Confidence Threshold

Mencari nilai threshold optimal yang memaksimalkan F1 Score kelas `TIDAK PANTAS`.  
Threshold default project: **0.75**

In [ ]:
# ─── Kalibrasi threshold ───
thresholds = np.arange(0.30, 0.96, 0.05)
results_thr = []

for thr in thresholds:
    preds_thr = (conf_scores >= thr).astype(int)
    results_thr.append({
        'threshold' : round(thr, 2),
        'f1'        : f1_score(all_labels, preds_thr, pos_label=1, average='binary', zero_division=0),
        'accuracy'  : accuracy_score(all_labels, preds_thr),
        'precision' : precision_score(all_labels, preds_thr, pos_label=1, average='binary', zero_division=0),
        'recall'    : recall_score(all_labels, preds_thr, pos_label=1, average='binary', zero_division=0)
    })

df_thr   = pd.DataFrame(results_thr)
best_row = df_thr.loc[df_thr['f1'].idxmax()]
OPTIMAL_THRESHOLD = float(best_row['threshold'])

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df_thr['threshold'], df_thr['f1'],        'b-o',  ms=5, lw=2, label='F1 TIDAK PANTAS')
ax.plot(df_thr['threshold'], df_thr['accuracy'],  'g--s', ms=5, lw=2, label='Accuracy')
ax.plot(df_thr['threshold'], df_thr['precision'], 'r-.^', ms=5, lw=2, label='Precision')
ax.plot(df_thr['threshold'], df_thr['recall'],    'm:d',  ms=5, lw=2, label='Recall')
ax.axvline(x=0.75,               color='orange',   ls='--', lw=2, label='Default (0.75)')
ax.axvline(x=OPTIMAL_THRESHOLD,  color='darkblue', ls=':',  lw=2, label=f'Optimal ({OPTIMAL_THRESHOLD:.2f})')
ax.axhline(y=0.82, color='red',   ls='--', lw=1.5, alpha=0.5, label='Target F1 ≥ 0.82')
ax.axhline(y=0.85, color='green', ls='--', lw=1.5, alpha=0.5, label='Target Acc ≥ 0.85')
ax.set_xlabel('Confidence Threshold')
ax.set_ylabel('Skor Metrik')
ax.set_title('Pengaruh Confidence Threshold terhadap Metrik', fontweight='bold')
ax.legend(loc='lower left', fontsize=9)
ax.set_xlim(0.28, 0.98)
ax.set_ylim(0, 1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('data/eval_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Tabel Kalibrasi Threshold:')
print(df_thr.to_string(index=False, float_format='%.4f'))

print(f'\n🎯 Threshold Optimal : {OPTIMAL_THRESHOLD:.2f}')
print(f'   F1 TIDAK PANTAS  : {best_row["f1"]:.4f}')
print(f'   Accuracy         : {best_row["accuracy"]:.4f}')
print(f'   Default project  : 0.75')

---
## 🚀 TAHAP 9 — Fungsi Inferensi `predict()` + Export

Membangun fungsi `predict(text)` yang akan dipanggil oleh Backend Engineer di FastAPI.

In [ ]:
# ─── Threshold yang digunakan ───
CONF_THRESHOLD = 0.75  # Sesuai spesifikasi project (ubah ke OPTIMAL_THRESHOLD jika lebih baik)


def predict(text: str, threshold: float = CONF_THRESHOLD) -> dict:
    """
    Memprediksi apakah teks termasuk PANTAS atau TIDAK PANTAS.

    Args:
        text      : Teks pesan pengguna
        threshold : Confidence threshold (default: 0.75)

    Returns:
        dict:
            label (str)          : 'PANTAS' atau 'TIDAK PANTAS'
            confidence (float)   : Skor kepercayaan model (0–1)
            is_appropriate (bool): True jika PANTAS
            action (str)         : Tindakan moderasi
            clean_text (str)     : Teks setelah preprocessing
    """
    # Preprocessing
    clean = preprocess_text(text, ALAY_MAP)

    if not clean:
        return {'label': 'PANTAS', 'confidence': 0.0,
                'is_appropriate': True, 'action': 'TAMPILKAN', 'clean_text': clean}

    # Tokenisasi
    enc = tokenizer(
        clean, max_length=MAX_LENGTH,
        padding='max_length', truncation=True, return_tensors='pt'
    )
    input_ids      = enc['input_ids'].to(DEVICE)
    attention_mask = enc['attention_mask'].to(DEVICE)
    token_type_ids = enc.get('token_type_ids', torch.zeros_like(input_ids)).to(DEVICE)

    # Inferensi
    model.eval()
    with torch.no_grad():
        out = model(input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids)

    probs    = torch.softmax(out.logits, dim=-1).cpu().numpy()[0]
    prob_bad = float(probs[1])

    # Tentukan label & tindakan berdasarkan threshold
    if prob_bad >= threshold:
        return {'label': 'TIDAK PANTAS', 'confidence': round(prob_bad, 4),
                'is_appropriate': False,
                'action': 'SEMBUNYIKAN_DAN_NOTIFIKASI',
                'clean_text': clean}
    elif prob_bad >= 0.5:
        return {'label': 'TIDAK PANTAS', 'confidence': round(prob_bad, 4),
                'is_appropriate': False,
                'action': 'SEMBUNYIKAN_DAN_TAWARKAN_REVIEW',
                'clean_text': clean}
    else:
        return {'label': 'PANTAS', 'confidence': round(float(probs[0]), 4),
                'is_appropriate': True,
                'action': 'TAMPILKAN',
                'clean_text': clean}


print('✅ Fungsi predict() siap!')
print(f'   Threshold default : {CONF_THRESHOLD}')

In [ ]:
# ─── Test fungsi predict() ───
test_cases = [
    'Halo semua, hari ini ada jadwal ujian pukul 09.00 di aula utama.',
    'Selamat pagi! Jangan lupa kerjakan tugas kelompok sebelum deadline.',
    'diskusinya sangat menarik, terima kasih atas masukannya.',
    'dasar bego lo! ga bisa ngerjain tugas gitu aja',
    'si kampret ini minta dihajar emang',
    'menyuruh berbuat vulgar karena kamu memang murahan',
    'anjir keren banget presentasinya bro!',
    'gila sih nilainya bagus banget, gue iri deh',
    'brengsek kenapa wifi kampus lemot banget sih'
]

print('🧪 HASIL PREDIKSI — AI SHIELD Inference Function')
print('=' * 72)
for i, text in enumerate(test_cases, 1):
    result = predict(text)
    icon   = '✅' if result['is_appropriate'] else '🚫'
    print(f'\n[{i:02d}] {icon} {result["label"]:15s} | Conf: {result["confidence"]:.4f} | {result["action"]}')
    print(f'      Teks Asli  : {text}')
    print(f'      Teks Bersih: {result["clean_text"]}')

In [ ]:
# ─── Benchmark kecepatan inferensi ───
bench_text = 'Hei teman-teman jangan lupa kumpulkan tugasnya besok pagi ya!'
N_RUNS = 30

# Warmup
for _ in range(5):
    predict(bench_text)

times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    predict(bench_text)
    times.append((time.perf_counter() - t0) * 1000)

print(f'⚡ BENCHMARK INFERENSI ({N_RUNS} runs | Device: {DEVICE})')
print(f'   Mean latency   : {statistics.mean(times):.1f} ms')
print(f'   Median latency : {statistics.median(times):.1f} ms')
print(f'   P95 latency    : {sorted(times)[int(N_RUNS*0.95)]:.1f} ms')

if statistics.mean(times) <= 500:
    print('   ✅ Memenuhi target latency < 500ms')
else:
    print('   ⚠️  Latency > 500ms. Pertimbangkan model quantization.')

In [ ]:
# ─── Export inference.py untuk Backend Engineer ───
inference_code = '''"""\nAI SHIELD — Inference Module\n============================\nModul ini digunakan oleh Backend Engineer (FastAPI).\nCara pakai: from inference import predict\n"""\nimport re, os, json, torch\nimport pandas as pd\nfrom transformers import BertTokenizer, BertForSequenceClassification\n\nMODEL_DIR = os.getenv(\'MODEL_DIR\', \'./indobert_aishield\')\nALAY_PATH  = os.getenv(\'ALAY_DICT_PATH\', \'./data/new_kamusalay.csv\')\nDEVICE     = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n\nwith open(f\'{MODEL_DIR}/model_config.json\') as f:\n    _CFG = json.load(f)\n\nMAX_LEN     = _CFG.get(\'max_length\', 128)\nLABELS      = _CFG.get(\'label_names\', [\'PANTAS\', \'TIDAK PANTAS\'])\nTHRESHOLD   = _CFG.get(\'confidence_threshold\', 0.75)\n\n_tok = BertTokenizer.from_pretrained(MODEL_DIR)\n_mdl = BertForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)\n_mdl.eval()\n\n_alay = {}\nif os.path.exists(ALAY_PATH):\n    _df = pd.read_csv(ALAY_PATH, encoding=\'latin-1\', header=None)\n    _alay = dict(zip(_df[0].str.lower(), _df[1]))\n\n\ndef _clean(text):\n    if not isinstance(text, str): return \'\'\n    text = text.lower()\n    text = re.sub(r\'\\brt\\b|@[\\w]+|https?://\\S+|www\\.\\S+|#|\\d+\', \' \', text)\n    text = text.encode(\'ascii\', \'ignore\').decode(\'ascii\')\n    text = re.sub(r\'[^\\w\\s]\', \' \', text)\n    if _alay: text = \' \'.join([_alay.get(w,w) for w in text.split()])\n    text = re.sub(r\'(.)\\1{2,}\', r\'\\1\\1\', text)\n    return re.sub(r\'\\s+\', \' \', text).strip()\n\n\ndef predict(text: str, threshold: float = THRESHOLD) -> dict:\n    clean = _clean(text)\n    if not clean:\n        return {\'label\': \'PANTAS\', \'confidence\': 0.0, \'is_appropriate\': True, \'action\': \'TAMPILKAN\'}\n    enc = _tok(clean, max_length=MAX_LEN, padding=\'max_length\', truncation=True, return_tensors=\'pt\')\n    ids = enc[\'input_ids\'].to(DEVICE)\n    msk = enc[\'attention_mask\'].to(DEVICE)\n    ttt = enc.get(\'token_type_ids\', torch.zeros_like(ids)).to(DEVICE)\n    with torch.no_grad():\n        out = _mdl(input_ids=ids, attention_mask=msk, token_type_ids=ttt)\n    probs = torch.softmax(out.logits, dim=-1).cpu().numpy()[0]\n    pb = float(probs[1])\n    if pb >= threshold:\n        return {\'label\': \'TIDAK PANTAS\', \'confidence\': round(pb,4), \'is_appropriate\': False, \'action\': \'SEMBUNYIKAN_DAN_NOTIFIKASI\'}\n    elif pb >= 0.5:\n        return {\'label\': \'TIDAK PANTAS\', \'confidence\': round(pb,4), \'is_appropriate\': False, \'action\': \'SEMBUNYIKAN_DAN_TAWARKAN_REVIEW\'}\n    return {\'label\': \'PANTAS\', \'confidence\': round(float(probs[0]),4), \'is_appropriate\': True, \'action\': \'TAMPILKAN\'}\n'''

with open('inference.py', 'w', encoding='utf-8') as f:
    f.write(inference_code)

print('✅ inference.py berhasil di-export!')
print()
print('📋 Cara pakai di FastAPI:')
print('   from inference import predict')
print("   result = predict('teks pesan pengguna')")
print("   if result['is_appropriate']:")
print('       # broadcast pesan')
print('   else:')
print('       # sembunyikan + notifikasi edukatif')

---
## ☁️ TAHAP 10 — Simpan Model ke Google Drive

Upload model ke Google Drive agar:
1. Model tidak hilang saat session Colab berakhir
2. Backend Engineer bisa mengunduh model
3. Link bisa dicantumkan di README project

In [ ]:
# ─── Upload ke Google Drive ───
# Uncomment dan jalankan cell ini untuk menyimpan ke Drive

from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/AI_SHIELD'
os.makedirs(f'{DRIVE_BASE}/model',     exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/inference', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/data',      exist_ok=True)

# Upload model
print('📤 Mengupload model ke Google Drive...')
shutil.copytree('./indobert_aishield', f'{DRIVE_BASE}/model/indobert_aishield', dirs_exist_ok=True)

# Upload inference.py
shutil.copy('./inference.py', f'{DRIVE_BASE}/inference/inference.py')

# Upload data CSV
for csv_file in ['train.csv', 'val.csv', 'test.csv', 'evaluation_results.json']:
    src = f'data/{csv_file}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_BASE}/data/{csv_file}')

print('✅ Semua file berhasil di-upload ke Google Drive!')
print(f'   📁 Lokasi : {DRIVE_BASE}')
print()
print('📌 Salin link folder Drive dan cantumkan di README project!')
print('   Cara: klik kanan folder di Drive → Get link → Copy link')

In [ ]:
# ─── Simpan hasil evaluasi ───
eval_results = {
    'accuracy'             : round(acc, 4),
    'f1_macro'             : round(f1_macro, 4),
    'f1_tidak_pantas'      : round(f1_bin, 4),
    'precision_macro'      : round(prec_macro, 4),
    'recall_macro'         : round(rec_macro, 4),
    'roc_auc'              : round(roc_auc, 4),
    'default_threshold'    : 0.75,
    'optimal_threshold'    : OPTIMAL_THRESHOLD,
    'false_positive_count' : int(fp),
    'false_negative_count' : int(fn),
    'test_size'            : len(all_labels)
}
with open('data/evaluation_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

# Simpan prediksi test set
df_result = df_test.copy()
df_result['prediction'] = all_predictions
df_result['confidence'] = conf_scores
df_result['pred_label'] = [MODEL_LABELS[p] for p in all_predictions]
df_result.to_csv('data/test_predictions.csv', index=False)

# ─── RINGKASAN AKHIR ───
print('\n' + '=' * 65)
print('🎉 AI SHIELD — PIPELINE MACHINE LEARNING SELESAI!')
print('=' * 65)
print()
print('📊 HASIL AKHIR:')
print(f'   Accuracy         : {acc:.4f}  (target ≥ 0.85 → {"✅" if acc>=0.85 else "❌"})')
print(f'   F1 TIDAK PANTAS  : {f1_bin:.4f}  (target ≥ 0.82 → {"✅" if f1_bin>=0.82 else "❌"})')
print(f'   ROC AUC          : {roc_auc:.4f}')
print(f'   Threshold Optimal: {OPTIMAL_THRESHOLD:.2f} (default: 0.75)')
print()
print('📁 FILE YANG DIHASILKAN:')
files = [
    ('data/train.csv',                'Dataset training (70%)'),
    ('data/val.csv',                  'Dataset validasi (15%)'),
    ('data/test.csv',                 'Dataset testing (15%)'),
    ('data/evaluation_results.json',  'Metrik evaluasi lengkap'),
    ('data/test_predictions.csv',     'Prediksi pada test set'),
    ('indobert_aishield/',            'Model IndoBERT fine-tuned'),
    ('inference.py',                  'Fungsi predict() untuk FastAPI'),
]
for fname, desc in files:
    status = '✅' if os.path.exists(fname) else '⚠️ '
    print(f'   {status} {fname:<40} → {desc}')
print()
print('🔗 LANGKAH SELANJUTNYA:')
print('   1. Upload model ke Google Drive → catat link di README')
print('   2. Serahkan inference.py ke Backend Engineer (FastAPI)')
print('   3. Deploy Streamlit demo ke Streamlit Cloud / HuggingFace Spaces')
print('=' * 65)